# LeetCode 1166 — Design File System bằng Python

Link đề: https://leetcode.com/problems/design-file-system/ · Độ khó: **Medium** · Chủ đề: Hash Table, String, Design, Trie

> **Bài Premium** — đề viết theo hiểu biết về bài (và bản mô tả công khai trên kho tham khảo doocs), nên đối chiếu lại nếu có tài khoản Premium. Chữ ký lớp dưới đây được suy ra từ lời giải Python công khai.

Skeleton LeetCode:

```python
class FileSystem:

    def __init__(self):
        ...

    def createPath(self, path: str, value: int) -> bool:
        ...

    def get(self, path: str) -> int:
        ...


# Your FileSystem object will be instantiated and called as such:
# obj = FileSystem()
# param_1 = obj.createPath(path,value)
# param_2 = obj.get(path)
```

Ghi chú: đây là bài **thiết kế lớp** (design). `createPath` trả `True/False`, `get` trả giá trị hoặc `-1`. Điểm cốt lõi: một đường dẫn chỉ được tạo khi **chưa tồn tại** và **đường dẫn cha đã tồn tại** (hoặc cha là gốc). Hai cách tự nhiên: **bảng băm** `path → value` + cắt chuỗi lấy cha, hoặc **Trie** theo từng thành phần thư mục.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: định dạng đường dẫn `/a/b/c`, khái niệm **đường dẫn cha** (parent path), điều kiện tạo thành công.
2. Nắm công cụ: `dict` (hash map), `str.rfind` và slicing để lấy cha, `str.split('/')`, cấu trúc **Trie** (cây tiền tố) bằng lớp và bằng dict lồng nhau.
3. Đi từ **danh sách tuần tự** → **hash map + rfind** → **Trie lớp `Node`** → **Trie dict lồng nhau**.
4. Biết cách mô phỏng lời gọi kiểu LeetCode (danh sách tên hàm + danh sách tham số) để test lớp thiết kế.
5. Tránh lỗi: tạo đè đường dẫn đã có, quên trường hợp cha là gốc, tự tạo luôn các thư mục cha (như `mkdir -p`), nhầm `get` trên thư mục trung gian chưa có giá trị.

## 2. Đề bài & ví dụ

Thiết kế hệ thống tệp cho phép **tạo đường dẫn mới** và gắn **giá trị** cho nó.

Đường dẫn có dạng một hoặc nhiều chuỗi con nối nhau, mỗi chuỗi con là `/` theo sau bởi **một hoặc nhiều chữ thường**. Ví dụ `"/leetcode"`, `"/leetcode/problems"` hợp lệ; `""` và `"/"` không hợp lệ.

Cài đặt lớp `FileSystem`:

- `bool createPath(string path, int value)`: tạo đường dẫn mới và gắn `value`, trả `true` nếu được. Trả `false` nếu đường dẫn **đã tồn tại** hoặc **đường dẫn cha chưa tồn tại**.
- `int get(string path)`: trả giá trị gắn với `path`, hoặc `-1` nếu đường dẫn không tồn tại.

**Ví dụ 1:**
```
["FileSystem","createPath","get"]
[[],["/a",1],["/a"]]
→ [null,true,1]
```

**Ví dụ 2:**
```
["FileSystem","createPath","createPath","get","createPath","get"]
[[],["/leet",1],["/leet/code",2],["/leet/code"],["/c/d",1],["/c"]]
→ [null,true,true,2,false,-1]
```
`"/c/d"` thất bại vì cha `"/c"` chưa tồn tại.

**Ràng buộc:** `2 <= path.length <= 100`; `1 <= value <= 10^9`; mọi đường dẫn hợp lệ, chỉ gồm chữ thường và `/`; tối đa `10^4` lời gọi.

In [1]:
import random
import time
from typing import Dict, List, Optional

examples = [
    (["FileSystem", "createPath", "get"],
     [[], ["/a", 1], ["/a"]],
     [None, True, 1]),
    (["FileSystem", "createPath", "createPath", "get", "createPath", "get"],
     [[], ["/leet", 1], ["/leet/code", 2], ["/leet/code"], ["/c/d", 1], ["/c"]],
     [None, True, True, 2, False, -1]),
]


def run_ops(cls, ops, args):
    """Mô phỏng cách LeetCode gọi lớp thiết kế: op đầu tiên là constructor."""
    obj = None
    out = []
    for op, a in zip(ops, args):
        if op == ops[0]:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, op)(*a))
    return out

for ops, args, want in examples:
    print(ops, "->", want)

['FileSystem', 'createPath', 'get'] -> [None, True, 1]
['FileSystem', 'createPath', 'createPath', 'get', 'createPath', 'get'] -> [None, True, True, 2, False, -1]


## 3. Kiến thức nền

### 3.1 Đường dẫn và đường dẫn cha

Đường dẫn `"/leet/code"` gồm các **thành phần** (component) `leet`, `code`. **Cha** của nó là `"/leet"` — bỏ thành phần cuối. Cha của `"/leet"` là **gốc** (chuỗi rỗng `""` theo quy ước của ta), luôn coi là tồn tại.

### 3.2 `str.rfind` và slicing

`path.rfind('/')` trả vị trí dấu `/` **cuối cùng**. Cắt `path[:i]` được cha. Với `"/a"`, `i = 0` → cha là `""` (gốc).

In [2]:
for p in ["/leet/code", "/leet", "/a/b/c"]:
    i = p.rfind("/")
    print(f"{p!r:12} rfind={i} parent={p[:i]!r} name={p[i+1:]!r}")
assert "/leet/code"[: "/leet/code".rfind("/")] == "/leet"
assert "/a"[: "/a".rfind("/")] == ""

'/leet/code' rfind=5 parent='/leet' name='code'
'/leet'      rfind=0 parent='' name='leet'
'/a/b/c'     rfind=4 parent='/a/b' name='c'


### 3.3 `str.split('/')`

`"/a/b".split('/')` cho `['', 'a', 'b']` — phần tử đầu rỗng vì chuỗi bắt đầu bằng `/`. Dùng `[1:]` để bỏ nó. Đây là cách lấy danh sách thành phần cho Trie.

In [3]:
print("/leet/code".split("/"), "/leet/code".split("/")[1:])
assert "/leet/code".split("/")[1:] == ["leet", "code"]

['', 'leet', 'code'] ['leet', 'code']


### 3.4 `dict` — bảng băm (hash map)

`dict` cho phép tra cứu, thêm, kiểm tra khoá trung bình `O(1)` (với khoá là chuỗi, thực tế `O(L)` để băm chuỗi độ dài `L`). `d.get(k, -1)` trả mặc định khi không có khoá — khớp đúng yêu cầu của `get`.

In [4]:
d = {"": -1, "/a": 1}
print("/a" in d, "/b" in d, d.get("/b", -1))
assert d.get("/b", -1) == -1

True False -1


### 3.5 Trie (cây tiền tố)

**Trie** là cây mà mỗi cạnh mang một "ký hiệu" (ở đây là **tên thư mục**, không phải từng ký tự). Đường từ gốc xuống một nút tương ứng một đường dẫn. Mỗi nút lưu `children: dict[str, Node]` và `value`. Ví dụ nhỏ: chèn `/a`, `/a/b`, `/c`.

In [5]:
class TinyNode:
    def __init__(self):
        self.children = {}

def show(node, indent=0):
    for name, child in sorted(node.children.items()):
        print("  " * indent + "/" + name)
        show(child, indent + 1)

root = TinyNode()
for p in ["/a", "/a/b", "/c"]:
    cur = root
    for part in p.split("/")[1:]:
        cur = cur.children.setdefault(part, TinyNode())
show(root)

/a
  /b
/c


### 3.6 Lớp (class) và mô phỏng lời gọi

Bài thiết kế được chấm bằng cách gọi lần lượt các phương thức. Hàm `run_ops` ở mục 2 mô phỏng điều đó: phần tử đầu là tên lớp (gọi constructor), các phần tử sau là tên phương thức gọi qua `getattr(obj, op)(*args)`.

In [6]:
class Counter1:
    def __init__(self):
        self.x = 0
    def inc(self, k):
        self.x += k
        return self.x

print(run_ops(Counter1, ["Counter1", "inc", "inc"], [[], [2], [3]]))
assert run_ops(Counter1, ["Counter1", "inc", "inc"], [[], [2], [3]]) == [None, 2, 5]

[None, 2, 5]


## 4. Ý tưởng nền tảng / trực giác

Mô phỏng tay ví dụ 2 với một "tập các đường dẫn đã có" (ban đầu chỉ có gốc `""`):

| Lời gọi | Cha | Cha có? | Path đã có? | Kết quả | Tập sau |
|---------|-----|---------|-------------|---------|---------|
| create `/leet`, 1 | `""` | có | không | `True` | {"", /leet} |
| create `/leet/code`, 2 | `/leet` | có | không | `True` | + /leet/code |
| get `/leet/code` | | | | `2` | |
| create `/c/d`, 1 | `/c` | **không** | | `False` | không đổi |
| get `/c` | | | | `-1` | |

Như vậy mỗi thao tác chỉ cần hai câu hỏi: "**path có chưa?**" và "**cha có chưa?**". Câu hỏi "có chưa" → nghĩ ngay đến bảng băm.

In [7]:
# Mô phỏng tay bằng set
existing = {""}
for p in ["/leet", "/leet/code", "/c/d"]:
    parent = p[: p.rfind("/")]
    ok = p not in existing and parent in existing
    if ok:
        existing.add(p)
    print(p, "parent", repr(parent), "->", ok)

/leet parent '' -> True
/leet/code parent '/leet' -> True
/c/d parent '/c' -> False


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** lưu một **danh sách** các cặp `(path, value)`. Mỗi thao tác quét tuyến tính danh sách để tìm path / cha.

In [8]:
class FileSystemList:
    def __init__(self):
        self.items = []              # list các (path, value)

    def _find(self, path):
        for p, v in self.items:
            if p == path:
                return v
        return None

    def createPath(self, path: str, value: int) -> bool:
        if self._find(path) is not None:
            return False
        parent = path[: path.rfind("/")]
        if parent and self._find(parent) is None:
            return False
        self.items.append((path, value))
        return True

    def get(self, path: str) -> int:
        v = self._find(path)
        return -1 if v is None else v

for ops, args, want in examples:
    got = run_ops(FileSystemList, ops, args)
    print(got)
    assert got == want

[None, True, 1]
[None, True, True, 2, False, -1]


**Độ phức tạp:** mỗi thao tác `O(k · L)` với `k` = số path đã tạo, `L` = độ dài path (so sánh chuỗi). Tổng `q` lời gọi: `O(q² · L)` = `10^8 · 100` tệ nhất — quá chậm.

**Nút thắt:** câu hỏi "path có tồn tại không" đang trả lời bằng quét tuyến tính. Cần cấu trúc tra cứu nhanh.

## 6. Bước đột phá

> Mỗi thao tác chỉ cần **tra cứu theo khoá chính xác** (path đầy đủ và path cha). Tra cứu theo khoá chính xác = **hash map**: trung bình `O(L)` mỗi lần (chi phí băm chuỗi).

Invariant: `paths` chứa đúng các đường dẫn đã tạo thành công, cùng với gốc `""` (giá trị giả `-1`).

Quy tắc:

- Nếu `path in paths` → `False` (không tạo đè).
- Nếu `parent not in paths` → `False`.
- Ngược lại `paths[path] = value`, trả `True`.
- `get` → `paths.get(path, -1)`; vì gốc `""` không bao giờ là input hợp lệ của `get`, giá trị giả của gốc không lộ ra.

Một góc nhìn khác: đường dẫn có **cấu trúc cây** (cha – con). Trie biểu diễn trực tiếp cấu trúc đó: tạo path = đi xuống theo các thành phần, cần mọi nút trung gian đã tồn tại, nút cuối chưa tồn tại.

In [9]:
paths = {"": -1}
def create(p, v):
    if p in paths or p[: p.rfind("/")] not in paths:
        return False
    paths[p] = v
    return True

print(create("/leet", 1), create("/leet/code", 2), create("/c/d", 1), create("/leet", 5), paths)

True True False False {'': -1, '/leet': 1, '/leet/code': 2}


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hash map `path → value` + `rfind`

In [10]:
class FileSystemHash:
    def __init__(self):
        self.paths = {"": -1}

    def createPath(self, path: str, value: int) -> bool:
        if path in self.paths:
            return False
        parent = path[: path.rfind("/")]
        if parent not in self.paths:
            return False
        self.paths[path] = value
        return True

    def get(self, path: str) -> int:
        return self.paths.get(path, -1)

for ops, args, want in examples:
    assert run_ops(FileSystemHash, ops, args) == want
print("hash OK")

hash OK


**Độ phức tạp:** mỗi thao tác `O(L)` trung bình (băm chuỗi, cắt chuỗi), tệ nhất lý thuyết `O(L · k)` nếu băm đụng độ nặng (thực tế không xảy ra). Bộ nhớ `O(tổng độ dài các path)`.

### 7.2 Cách 3 — Trie với lớp `Node`

Mỗi nút có `children` (dict tên → nút) và `value` (`-1` nghĩa là… ở đây mọi nút trong trie đều là path đã tạo, vì ta không bao giờ tạo nút trung gian; vẫn lưu value cho từng nút).

`insert(parts, value)`: đi qua các thành phần trừ cái cuối — nếu thiếu nút thì thất bại (cha không tồn tại). Ở thành phần cuối: nếu đã có → thất bại; ngược lại tạo nút.

In [11]:
class Node:
    __slots__ = ("children", "value")

    def __init__(self, value=-1):
        self.children = {}
        self.value = value


class FileSystemTrie:
    def __init__(self):
        self.root = Node()

    def createPath(self, path: str, value: int) -> bool:
        parts = path.split("/")[1:]
        cur = self.root
        for name in parts[:-1]:
            if name not in cur.children:
                return False             # cha (hoặc tổ tiên) chưa tồn tại
            cur = cur.children[name]
        last = parts[-1]
        if last in cur.children:
            return False                 # path đã tồn tại
        cur.children[last] = Node(value)
        return True

    def get(self, path: str) -> int:
        cur = self.root
        for name in path.split("/")[1:]:
            if name not in cur.children:
                return -1
            cur = cur.children[name]
        return cur.value

for ops, args, want in examples:
    assert run_ops(FileSystemTrie, ops, args) == want
print("trie OK")

trie OK


**Độ phức tạp:** mỗi thao tác `O(L)` (split + đi xuống tối đa `L/2` nút). Bộ nhớ: mỗi path tạo **một** nút mới, các tiền tố chung được **chia sẻ** → tiết kiệm khi nhiều path chung thư mục cha dài.

### 7.3 Cách 4 — Trie bằng dict lồng nhau

Gọn hơn: mỗi nút là một `dict`; khoá đặc biệt `"#"` (không phải chữ thường nên không trùng tên thư mục) lưu giá trị.

In [12]:
class FileSystemNestedDict:
    VAL = "#"

    def __init__(self):
        self.root = {}

    def createPath(self, path: str, value: int) -> bool:
        *parents, last = path.split("/")[1:]
        cur = self.root
        for name in parents:
            cur = cur.get(name)
            if cur is None:
                return False
        if last in cur:
            return False
        cur[last] = {self.VAL: value}
        return True

    def get(self, path: str) -> int:
        cur = self.root
        for name in path.split("/")[1:]:
            cur = cur.get(name)
            if cur is None:
                return -1
        return cur[self.VAL]

for ops, args, want in examples:
    assert run_ops(FileSystemNestedDict, ops, args) == want
print("nested dict OK")

nested dict OK


Cú pháp `*parents, last = [...]` là **unpacking mở rộng**: `last` nhận phần tử cuối, `parents` nhận list các phần tử còn lại.

**Độ phức tạp:** như cách 3.

## 8. TRACE từng bước

Trace cách 2 (hash map) trên ví dụ 2, in trạng thái `paths` sau mỗi lời gọi.

| # | Lời gọi | parent | path in? | parent in? | trả về |
|---|---------|--------|----------|------------|--------|
| 1 | create /leet 1 | "" | F | T | True |
| 2 | create /leet/code 2 | /leet | F | T | True |
| 3 | get /leet/code | | | | 2 |
| 4 | create /c/d 1 | /c | F | F | False |
| 5 | get /c | | | | -1 |

In [13]:
def trace_hash(ops, args):
    fs = None
    for op, a in zip(ops, args):
        if op == "FileSystem":
            fs = FileSystemHash()
            print("init  paths =", fs.paths)
            continue
        if op == "createPath":
            p = a[0]
            parent = p[: p.rfind("/")]
            info = f"parent={parent!r} path_in={p in fs.paths} parent_in={parent in fs.paths}"
        else:
            info = ""
        res = getattr(fs, op)(*a)
        print(f"{op:10} {a!s:22} {info:45} -> {res!s:5} paths={fs.paths}")

trace_hash(*examples[1][:2])

init  paths = {'': -1}
createPath ['/leet', 1]           parent='' path_in=False parent_in=True        -> True  paths={'': -1, '/leet': 1}
createPath ['/leet/code', 2]      parent='/leet' path_in=False parent_in=True   -> True  paths={'': -1, '/leet': 1, '/leet/code': 2}
get        ['/leet/code']                                                       -> 2     paths={'': -1, '/leet': 1, '/leet/code': 2}
createPath ['/c/d', 1]            parent='/c' path_in=False parent_in=False     -> False paths={'': -1, '/leet': 1, '/leet/code': 2}
get        ['/c']                                                               -> -1    paths={'': -1, '/leet': 1, '/leet/code': 2}


Trace cách 3 (Trie): in cây sau mỗi lần tạo thành công.

In [14]:
def dump(node, name="(root)", depth=0):
    print("    " * depth + f"{name} [{node.value}]")
    for k in sorted(node.children):
        dump(node.children[k], k, depth + 1)

def trace_trie(calls):
    fs = FileSystemTrie()
    for p, v in calls:
        ok = fs.createPath(p, v)
        print(f"createPath({p!r}, {v}) -> {ok}")
        if ok:
            dump(fs.root)

trace_trie([("/a", 1), ("/a/b", 2), ("/a/b/c", 3), ("/x/y", 4), ("/a/d", 5), ("/a/b", 9)])

createPath('/a', 1) -> True
(root) [-1]
    a [1]
createPath('/a/b', 2) -> True
(root) [-1]
    a [1]
        b [2]
createPath('/a/b/c', 3) -> True
(root) [-1]
    a [1]
        b [2]
            c [3]
createPath('/x/y', 4) -> False
createPath('/a/d', 5) -> True
(root) [-1]
    a [1]
        b [2]
            c [3]
        d [5]
createPath('/a/b', 9) -> False


## 9. Code nộp LeetCode

Chọn cách 2 (hash map) — ngắn nhất, `O(L)` mỗi lời gọi.

In [15]:
class FileSystem:

    def __init__(self):
        self.paths = {"": -1}            # gốc luôn tồn tại

    def createPath(self, path: str, value: int) -> bool:
        if path in self.paths:
            return False
        parent = path[: path.rfind("/")]
        if parent not in self.paths:
            return False
        self.paths[path] = value
        return True

    def get(self, path: str) -> int:
        return self.paths.get(path, -1)


# Your FileSystem object will be instantiated and called as such:
# obj = FileSystem()
# param_1 = obj.createPath(path,value)
# param_2 = obj.get(path)

for ops, args, want in examples:
    assert run_ops(FileSystem, ops, args) == want
print("FileSystem OK")

FileSystem OK


Ghi chú: kiểm tra `class Solution` không áp dụng cho bài thiết kế; tên lớp đúng theo LeetCode là `FileSystem`.

In [16]:
# Một lớp Solution "bọc" để tiện chạy chuỗi lệnh kiểu LeetCode (không cần khi nộp)
class Solution:
    def simulate(self, ops: List[str], args: List[list]) -> list:
        return run_ops(FileSystem, ops, args)

print(Solution().simulate(*examples[1][:2]))

[None, True, True, 2, False, -1]


## 10. Vì sao đúng

**Invariant:** sau mỗi lời gọi, tập khoá của `paths` (trừ `""`) đúng bằng tập các đường dẫn đã được tạo thành công, và mỗi khoá ánh xạ tới giá trị được truyền khi tạo.

- **Khởi tạo:** chỉ có `""` — chưa tạo path nào. Đúng.
- **createPath:** chỉ thêm khoá khi (1) chưa tồn tại — không ghi đè, (2) cha tồn tại. Cha của path có một thành phần là `""` (luôn có). Đúng với đặc tả. Nếu thất bại, không thay đổi gì.
- **get:** tra cứu trực tiếp; path hợp lệ không bao giờ là `""` nên không trả nhầm giá trị gốc.

**Hệ quả quan trọng:** vì chỉ được tạo khi cha đã tồn tại, mọi tổ tiên của một path đã tạo cũng đã tạo (quy nạp theo số thành phần). Do đó Trie không bao giờ cần "nút trung gian không có giá trị" — điều này làm cho cách 3/4 đúng: đi xuống theo `parts[:-1]` thành công ⟺ cha tồn tại.

## 11. Time / Space Complexity

Gọi `L` = độ dài path (≤ 100), `q` = số lời gọi (≤ `10^4`), `S` = tổng độ dài các path đã tạo.

| Cách | createPath | get | Space |
|------|-----------|-----|-------|
| 1. List | `O(k·L)` | `O(k·L)` | `O(S)` |
| 2. Hash map | `O(L)` TB | `O(L)` TB | `O(S)` (lưu nguyên chuỗi path, tiền tố bị lặp) |
| 3. Trie Node | `O(L)` | `O(L)` | `O(S)` tệ nhất, thường ít hơn nhờ chia sẻ tiền tố |
| 4. Trie dict | `O(L)` | `O(L)` | như 3 |

`k` = số path đã tạo tại thời điểm gọi. TB = trung bình (hash).

## 12. Bộ test

### 12.1 Ví dụ + edge case

In [17]:
impls = {
    "list": FileSystemList,
    "hash": FileSystemHash,
    "trie": FileSystemTrie,
    "nested": FileSystemNestedDict,
    "FileSystem": FileSystem,
}
edge = [
    # tạo đè
    (["FileSystem", "createPath", "createPath", "get"], [[], ["/a", 1], ["/a", 2], ["/a"]], [None, True, False, 1]),
    # không tự tạo cha (không giống mkdir -p)
    (["FileSystem", "createPath", "get", "get"], [[], ["/a/b/c", 1], ["/a"], ["/a/b/c"]], [None, False, -1, -1]),
    # tên trùng ở các nhánh khác nhau
    (["FileSystem", "createPath", "createPath", "createPath", "createPath", "get", "get"],
     [[], ["/a", 1], ["/b", 2], ["/a/x", 3], ["/b/x", 4], ["/a/x"], ["/b/x"]],
     [None, True, True, True, True, 3, 4]),
    # tiền tố chuỗi nhưng không phải cha: "/ab" không phải cha của "/a/b"
    (["FileSystem", "createPath", "createPath"], [[], ["/ab", 1], ["/a/b", 2]], [None, True, False]),
    # get trên path chưa tạo nhưng là tiền tố
    (["FileSystem", "createPath", "get"], [[], ["/leet", 1], ["/lee"]], [None, True, -1]),
]
for name, cls in impls.items():
    for ops, args, want in examples + edge:
        assert run_ops(cls, ops, args) == want, (name, ops, args)
print("all impls pass", len(examples + edge))

all impls pass 7


### 12.2 Đối chiếu ngẫu nhiên (seed cố định)

Sinh path từ bảng tên nhỏ `{a, b, ab}` với độ sâu 1–4 để thường xuyên có cha/không cha, tạo đè, tiền tố chuỗi. So mọi cách với `FileSystemList` (baseline).

In [18]:
def random_ops(rng, q):
    names = ["a", "b", "ab"]
    ops, args = ["FileSystem"], [[]]
    for _ in range(q):
        depth = rng.randint(1, 4)
        p = "".join("/" + rng.choice(names) for _ in range(depth))
        if rng.random() < 0.6:
            ops.append("createPath"); args.append([p, rng.randint(1, 10**9)])
        else:
            ops.append("get"); args.append([p])
    return ops, args

rng = random.Random(1166)
for _ in range(300):
    ops, args = random_ops(rng, rng.randint(1, 40))
    want = run_ops(FileSystemList, ops, args)
    for name, cls in impls.items():
        assert run_ops(cls, ops, args) == want, name
print("random cross-check OK")

random cross-check OK


### 12.3 Test giới hạn

`10^4` lời gọi: xây chuỗi sâu `/a/a/a/...` (độ dài ~100) và nhiều nhánh rộng.

In [19]:
ops, args = ["FileSystem"], [[]]
p = ""
for i in range(49):                 # path dài tối đa ~98 ký tự
    p += "/a"
    ops.append("createPath"); args.append([p, i + 1])
rng = random.Random(3)
for i in range(10**4 - 50):
    name = "".join(rng.choice("abcdefgh") for _ in range(rng.randint(1, 8)))
    if i % 2:
        ops.append("createPath"); args.append(["/a/" + name, i])
    else:
        ops.append("get"); args.append(["/a/" + name])
for name in ["hash", "trie", "nested", "FileSystem"]:
    t0 = time.perf_counter()
    res = run_ops(impls[name], ops, args)
    print(name, sum(r is True for r in res), f"{time.perf_counter() - t0:.3f}s")
assert run_ops(FileSystem, ops, args) == run_ops(FileSystemTrie, ops, args)

hash 3594 0.004s


trie 3594 0.010s


nested 3594 0.008s
FileSystem 3594 0.004s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Ghi đè path đã tồn tại

In [20]:
class WrongOverwrite(FileSystemHash):
    def createPath(self, path, value):
        parent = path[: path.rfind("/")]
        if parent not in self.paths:
            return False
        self.paths[path] = value          # quên kiểm tra path đã có
        return True

ops, args = ["FileSystem", "createPath", "createPath", "get"], [[], ["/a", 1], ["/a", 2], ["/a"]]
print("sai:", run_ops(WrongOverwrite, ops, args), "| đúng:", run_ops(FileSystem, ops, args))
assert run_ops(WrongOverwrite, ops, args) != run_ops(FileSystem, ops, args)

sai: [None, True, True, 2] | đúng: [None, True, False, 1]


**Vì sao sai:** đặc tả nói path đã tồn tại → `false` và giữ giá trị cũ. **Sửa:** kiểm tra `path in self.paths` trước.

### Lỗi 2 — Quên gốc: không khởi tạo `""`

In [21]:
class WrongNoRoot:
    def __init__(self):
        self.paths = {}                    # thiếu "" → path cấp 1 không có cha
    createPath = FileSystemHash.createPath
    get = FileSystemHash.get

ops, args = examples[0][:2]
print("sai:", run_ops(WrongNoRoot, ops, args), "| đúng:", examples[0][2])
assert run_ops(WrongNoRoot, ops, args) != examples[0][2]

sai: [None, False, -1] | đúng: [None, True, 1]


**Vì sao sai:** cha của `"/a"` là `""`, không có trong dict → mọi path cấp 1 đều thất bại → không tạo được gì. **Sửa:** `self.paths = {"": -1}` (hoặc `if parent and parent not in ...`).

### Lỗi 3 — Kiểm tra cha bằng `startswith` (nhầm tiền tố chuỗi với cha)

In [22]:
class WrongStartswith(FileSystemHash):
    def createPath(self, path, value):
        if path in self.paths:
            return False
        if path.count("/") > 1 and not any(path.startswith(p) for p in self.paths if p):
            return False
        self.paths[path] = value
        return True

ops, args = ["FileSystem", "createPath", "createPath"], [[], ["/ab", 1], ["/a/b", 2]]
ops2, args2 = ["FileSystem", "createPath", "createPath"], [[], ["/a", 1], ["/a/b/c", 2]]
print("sai:", run_ops(WrongStartswith, ops2, args2), "| đúng:", run_ops(FileSystem, ops2, args2))
assert run_ops(WrongStartswith, ops2, args2) != run_ops(FileSystem, ops2, args2)

sai: [None, True, True] | đúng: [None, True, False]


**Vì sao sai:** "có một tổ tiên tồn tại" khác với "**cha trực tiếp** tồn tại": `/a/b/c` cần `/a/b`, không chỉ `/a`. Ngoài ra `startswith("/a")` còn đúng với `"/ab..."`. **Sửa:** lấy cha chính xác bằng `rfind`.

### Lỗi 4 — Trie tự tạo nút trung gian (kiểu `mkdir -p`)

In [23]:
class WrongAutoCreate(FileSystemTrie):
    def createPath(self, path, value):
        cur = self.root
        parts = path.split("/")[1:]
        for name in parts[:-1]:
            cur = cur.children.setdefault(name, Node())   # tự tạo cha!
        if parts[-1] in cur.children:
            return False
        cur.children[parts[-1]] = Node(value)
        return True

ops, args = examples[1][:2]
print("sai:", run_ops(WrongAutoCreate, ops, args), "| đúng:", examples[1][2])
assert run_ops(WrongAutoCreate, ops, args) != examples[1][2]

sai: [None, True, True, 2, True, -1] | đúng: [None, True, True, 2, False, -1]


**Vì sao sai:** `setdefault` tạo ngầm `/c`, nên `createPath("/c/d")` trả `True` và `get("/c")` trả `-1` nhưng nút `/c` đã tồn tại sai lệch (lần sau `createPath("/c", ...)` sẽ báo đã tồn tại). **Sửa:** khi đi qua cha, thiếu nút → trả `False` ngay.

## 14. Mẹo & nhận dạng pattern

- **Thao tác chỉ tra cứu khoá chính xác** → hash map là đủ, đừng vội dùng Trie.
- **Cần thao tác theo tiền tố** (liệt kê con, xoá cả cây con, đếm path có tiền tố) → Trie phát huy tác dụng.
- Đường dẫn/URL/namespace có cấu trúc phân cấp → Trie theo **thành phần** (component), không theo ký tự.
- Quy ước gốc là `""` giúp xoá trường hợp đặc biệt "path cấp 1".
- Bài design: luôn viết hàm mô phỏng `run_ops` để test toàn bộ chuỗi lệnh.

## 15. Biến thể

1. **LC 588 Design In-Memory File System** (Hard, Premium) — `ls`, `mkdir -p`, `addContentToFile`, `readContentFromFile`: cần Trie vì phải liệt kê con theo thứ tự từ điển.
2. **LC 1233 Remove Sub-Folders from the Filesystem** — loại thư mục con: sort hoặc Trie.
3. **LC 208 Implement Trie** — Trie theo ký tự.
4. Thêm `delete(path)`: xoá path **và mọi path con**. Với Trie: xoá cạnh tới nút. Với hash map: phải quét mọi khoá có tiền tố `path + "/"` → `O(k·L)`.
5. Thêm `countChildren(path)` / `listChildren(path)` — Trie trả lời ngay.

In [24]:
# Biến thể 4 + 5 trên Trie
class FileSystemPlus(FileSystemTrie):
    def _node(self, path):
        cur = self.root
        for name in path.split("/")[1:]:
            cur = cur.children.get(name)
            if cur is None:
                return None
        return cur

    def delete(self, path: str) -> bool:
        *parents, last = path.split("/")[1:]
        parent = self._node("/" + "/".join(parents)) if parents else self.root
        if parent is None or last not in parent.children:
            return False
        del parent.children[last]
        return True

    def listChildren(self, path: str) -> list:
        node = self._node(path)
        return sorted(node.children) if node else []

fs = FileSystemPlus()
for p, v in [("/a", 1), ("/a/x", 2), ("/a/y", 3), ("/a/x/z", 4)]:
    fs.createPath(p, v)
print(fs.listChildren("/a"), fs.delete("/a/x"), fs.get("/a/x/z"), fs.listChildren("/a"))
assert fs.get("/a/x/z") == -1 and fs.listChildren("/a") == ["y"]

['x', 'y'] True -1 ['y']


## 16. So sánh các cách

| Cách | Ý tưởng | Time / lời gọi | Space | Khi nào dùng |
|------|---------|----------------|-------|--------------|
| 1. List | Quét tuyến tính | `O(k·L)` | `O(S)` | Kiểm chứng |
| 2. Hash map | `path → value`, cha bằng `rfind` | `O(L)` TB | `O(S)` | Nộp bài, ngắn nhất |
| 3. Trie Node | Cây theo thành phần | `O(L)` | ≤ `O(S)` | Cần thao tác cây con |
| 4. Trie dict | Như 3, dict lồng | `O(L)` | ≤ `O(S)` | Code gọn, ít lớp |

## 17. Bài tập vận dụng

**Bài 1 (đoán output):** chuỗi lệnh `create /x 5`, `create /x/y 6`, `create /x 7`, `get /x`, `get /x/y/z` cho kết quả gì?

**Bài 2 (tính tay):** sau các lệnh `create /a 1`, `create /a/b 2`, `create /a/c 3`, `create /a/b/d 4`, vẽ Trie.

**Bài 3 (điền TODO):** hàm `parent_of(path)` trả cha (gốc là `""`) không dùng `rfind` (dùng `split`/`join`).

**Bài 4 (tự viết lại):** viết `FileSystem` chỉ dùng `set` các path + `dict` giá trị riêng (hai cấu trúc), đúng mọi test.

**Bài 5 (mini challenge):** thêm phương thức `createPathRecursive(path, value)` hoạt động như `mkdir -p`: tạo luôn các cha thiếu với giá trị `0`, trả `False` chỉ khi path đã tồn tại.

In [25]:
# Bài 3 — TODO
def parent_of(path: str) -> str:
    # TODO
    raise NotImplementedError

# assert parent_of("/a/b/c") == "/a/b"
# assert parent_of("/a") == ""

In [26]:
# Bài 5 — TODO
class FileSystemRecursive(FileSystem):
    def createPathRecursive(self, path: str, value: int) -> bool:
        # TODO
        pass

# fs = FileSystemRecursive()
# assert fs.createPathRecursive("/a/b/c", 7) is True
# assert fs.get("/a") == 0 and fs.get("/a/b/c") == 7
# assert fs.createPathRecursive("/a/b", 1) is False

## 18. Gợi ý 3 mức

- **Bài 1:** Mức 1: lần tạo `/x` thứ hai? Mức 2: đã tồn tại → `False`, giữ 5. Mức 3: `/x/y/z` chưa tạo → `-1`.
- **Bài 2:** Mức 1: gốc có một con `a`. Mức 2: `a` có hai con `b`, `c`. Mức 3: `b` có con `d`.
- **Bài 3:** Mức 1: `split("/")`. Mức 2: bỏ phần tử cuối. Mức 3: `"/".join(...)` — với `"/a"` cho `""`.
- **Bài 4:** Mức 1: `exists = {""}`. Mức 2: `values = {}`. Mức 3: `get` dùng `values.get(path, -1)`.
- **Bài 5:** Mức 1: duyệt các tiền tố `/a`, `/a/b`, … Mức 2: tiền tố chưa có → gán 0. Mức 3: kiểm tra path cuối đã tồn tại **trước** khi tạo cha.

## 19. Đáp án tham khảo

In [27]:
# Bài 1
ops = ["FileSystem", "createPath", "createPath", "createPath", "get", "get"]
args = [[], ["/x", 5], ["/x/y", 6], ["/x", 7], ["/x"], ["/x/y/z"]]
assert run_ops(FileSystem, ops, args) == [None, True, True, False, 5, -1]

# Bài 2
fs = FileSystemTrie()
for p, v in [("/a", 1), ("/a/b", 2), ("/a/c", 3), ("/a/b/d", 4)]:
    assert fs.createPath(p, v)
dump(fs.root)

# Bài 3
def parent_of_sol(path: str) -> str:
    return "/".join(path.split("/")[:-1])

assert parent_of_sol("/a/b/c") == "/a/b"
assert parent_of_sol("/a") == ""

# Bài 4
class FileSystemTwoSets:
    def __init__(self):
        self.exists = {""}
        self.values = {}
    def createPath(self, path, value):
        if path in self.exists or parent_of_sol(path) not in self.exists:
            return False
        self.exists.add(path); self.values[path] = value
        return True
    def get(self, path):
        return self.values.get(path, -1)

rng = random.Random(44)
for _ in range(200):
    ops_, args_ = random_ops(rng, 30)
    assert run_ops(FileSystemTwoSets, ops_, args_) == run_ops(FileSystem, ops_, args_)
print("bài 1-4 OK")

(root) [-1]
    a [1]
        b [2]
            d [4]
        c [3]
bài 1-4 OK


In [28]:
# Bài 5
class FileSystemRecursiveSol(FileSystem):
    def createPathRecursive(self, path: str, value: int) -> bool:
        if path in self.paths:
            return False
        idx = path.find("/", 1)
        while idx != -1:
            prefix = path[:idx]
            if prefix not in self.paths:
                self.paths[prefix] = 0
            idx = path.find("/", idx + 1)
        self.paths[path] = value
        return True

fs = FileSystemRecursiveSol()
assert fs.createPathRecursive("/a/b/c", 7) is True
assert fs.get("/a") == 0 and fs.get("/a/b") == 0 and fs.get("/a/b/c") == 7
assert fs.createPathRecursive("/a/b", 1) is False
assert fs.createPath("/a/b/c/d", 9) is True
print("bài 5 OK")

bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
class FileSystem:
    def __init__(self):
        self.paths = {"": -1}                   # gốc
    def createPath(self, path, value):
        if path in self.paths: return False      # đã có
        if path[:path.rfind('/')] not in self.paths: return False   # thiếu cha
        self.paths[path] = value
        return True
    def get(self, path):
        return self.paths.get(path, -1)
```

**Checklist:**

- [ ] Có khởi tạo gốc `""` (hoặc xử lý riêng path cấp 1)?
- [ ] Path đã tồn tại → `False` và **không** ghi đè?
- [ ] Cha lấy **chính xác** bằng cắt tới `/` cuối (không dùng `startswith`)?
- [ ] Không tự tạo thư mục cha?
- [ ] `get` path không tồn tại → `-1`?
- [ ] Tên lớp đúng `FileSystem`, phương thức `createPath`, `get`?